# Sonar Mine Detection: Exploratory Data Analysis

**Context:** Submarine warfare mine detection using sonar signal classification

**Critical Constraint:** Missing a single mine can be catastrophic (ship/submarine loss)

**Primary Objective:** Maximize recall (detect all mines) while maintaining acceptable precision

---

This notebook explores the sonar dataset to understand:
- Data quality and completeness
- Feature characteristics and distributions  
- Class separability patterns
- Preprocessing requirements
- Modeling strategy constraints

## 1. Problem Definition

### The Stakes

In mine detection:
- **False Negative (missed mine):** Ship destroyed, lives lost, mission failure
- **False Positive (false alarm):** Investigation cost, time delay

Cost asymmetry: FN >> FP (by orders of magnitude)

### Success Criteria

1. **Primary metric:** F2 score (weights recall 2x higher than precision)
2. **Target:** >95% recall on test set
3. **Stretch goal:** 100% recall through threshold optimization

### Dataset

- **Source:** UCI Machine Learning Repository - Sonar Dataset
- **Samples:** 208 sonar bounces
- **Features:** 60 frequency bands (0-1 range)
- **Target:** Rock (R) vs Metal Cylinder/Mine (M)
- **Challenge:** Small dataset - overfitting risk

In [ ]:
# Imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats

import sys
sys.path.insert(0, '..')

from configs.config import (
    DATA_RAW_PATH, OUTCOME_MAPPING, DEFAULT_SEED, TEST_SIZE
)

from ml_toolkit.eda import (
    load_dataset, correlation_with_target, count_high_correlations,
    EconomistStyler, RadarChartFactory, KDEPlotFactory, OutlierPlotFactory,
    detect_outliers_iqr, detect_outliers_zscore
)

# Configure Economist styling
EconomistStyler.configure()

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print('✓ Environment configured')
print(f'Random seed: {DEFAULT_SEED}')
print(f'Test size: {TEST_SIZE*100:.0f}%')

## 2. Data Loading

In [ ]:
# Load complete dataset
df, schema = load_dataset(path=DATA_RAW_PATH, outcome_mapping=OUTCOME_MAPPING)

print(f"Dataset: {schema['n_samples']} samples × {schema['n_features']} features")
print(f"Target: {schema['target_name']} ({schema['target_values']})")
print(f"\nOutcome mapping: {OUTCOME_MAPPING}")

df.head()

## 3. Train/Test Split (Data Leakage Prevention)

**Critical:** Split data BEFORE any analysis to prevent data leakage.

All EDA performed on training set only. Test set remains locked until final model evaluation.

In [ ]:
# Separate features and target
y = df['outcome']
X = df.drop('outcome', axis=1)

# Stratified split to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    random_state=DEFAULT_SEED, 
    stratify=y
)

# Recombine for EDA
df_train = X_train.copy()
df_train['outcome'] = y_train

df_test = X_test.copy()
df_test['outcome'] = y_test

print(f"Training set: {len(df_train)} samples ({(1-TEST_SIZE)*100:.0f}%)")
print(f"Test set: {len(df_test)} samples ({TEST_SIZE*100:.0f}%)")
print(f"\n⚠️ ALL ANALYSIS PERFORMED ON TRAINING DATA ONLY")
print(f"Test set remains locked until final evaluation")

## 4. Missing Value Analysis

Comprehensive check for data completeness.

In [ ]:
# Missing value analysis
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Pct': missing_pct
}).sort_values('Missing_Count', ascending=False)

total_missing = missing_df['Missing_Count'].sum()

print(f"Total missing values: {total_missing}")
print(f"Features with missing data: {(missing > 0).sum()}")

if total_missing > 0:
    print(f"\nFeatures with missing values:")
    print(missing_df[missing_df['Missing_Count'] > 0].head(10))
else:
    print("\n✓ No missing values detected")
    print("Dataset is complete - no imputation required")

print(f"\n**Decision:** {'Imputation strategy needed' if total_missing > 0 else 'Proceed with complete cases'}")

## 5. Class Balance Analysis

Assess target distribution and establish baseline accuracy.

In [ ]:
# Class distribution
class_counts = df_train['outcome'].value_counts().sort_index()
class_props = class_counts / len(df_train)
baseline_acc = class_counts.max() / class_counts.sum()

print("Class Distribution (Training Set):")
print(f"  Rocks (0): {class_counts[0]} samples ({class_props[0]:.1%})")
print(f"  Mines (1): {class_counts[1]} samples ({class_props[1]:.1%})")
print(f"\nBaseline accuracy: {baseline_acc:.2%} (always predict majority class)")
print(f"**Our model must exceed {baseline_acc:.2%} to be useful**")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#0F5499', '#E3120B']  # Blue for Rock, Red for Mine
class_counts.plot(kind='bar', ax=axes[0], color=colors, alpha=0.8)
axes[0].set_title('Class Distribution', fontsize=14, weight='bold')
axes[0].set_xlabel('Class', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_xticklabels(['Rock (0)', 'Mine (1)'], rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
axes[1].pie(class_counts, labels=['Rock', 'Mine'], colors=colors, 
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Proportion', fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

print(f"\n**Insight:** Classes are {'balanced' if abs(class_props[0] - 0.5) < 0.1 else 'imbalanced'}")
print(f"**Decision:** {'Standard classification' if abs(class_props[0] - 0.5) < 0.1 else 'Consider class weights or resampling'}")

## 6. Feature Distributions by Class (KDE Analysis)

Examine how feature distributions differ between rocks and mines.
Good separation indicates discriminative features.

In [ ]:
# Get top features by correlation for focused analysis
feature_cols = [c for c in df_train.columns if c != 'outcome']
correlations = df_train[feature_cols + ['outcome']].corr()['outcome'].abs().sort_values(ascending=False)
top_features = correlations.iloc[1:13].index.tolist()  # Top 12 features

print("Top 12 features by absolute correlation with target:")
for i, feat in enumerate(top_features, 1):
    print(f"  {i}. {feat}: {correlations[feat]:.3f}")

# Create KDE plots
fig, axes = KDEPlotFactory.create_class_comparison(
    data=df_train,
    features=top_features,
    target_col='outcome',
    class_0_label='Rock',
    class_1_label='Mine',
    n_cols=3,
    figsize=(15, 12)
)

plt.suptitle('Feature Distributions by Class (Top 12 Features)', 
            fontsize=16, weight='bold', y=1.00)
plt.show()

print("\n**Insight:** Features with non-overlapping distributions provide strong discriminative power")
print("**Decision:** Multiple features show class separation - classification is feasible")

## 7. Summary Statistics

In [ ]:
# Summary statistics for top features
summary = df_train[top_features].describe().T
summary['range'] = summary['max'] - summary['min']
summary['cv'] = summary['std'] / summary['mean']  # Coefficient of variation

print("Summary Statistics (Top 12 Features):")
print(summary[['mean', 'std', 'min', 'max', 'range', 'cv']].round(4))

## 8. Outlier Analysis

Detect extreme values that may impact model training.

In [ ]:
# Outlier detection using IQR method
outlier_counts = {}
for feat in top_features[:6]:  # Check top 6 features
    outliers_iqr = detect_outliers_iqr(df_train[feat])
    outliers_zscore = detect_outliers_zscore(df_train[feat], threshold=3)
    outlier_counts[feat] = {
        'IQR': outliers_iqr.sum(),
        'Z-score': outliers_zscore.sum()
    }

outlier_df = pd.DataFrame(outlier_counts).T
print("Outlier Counts (Top 6 Features):")
print(outlier_df)

total_outliers = outlier_df['IQR'].sum()
print(f"\nTotal outliers (IQR method): {total_outliers}")

# Box plots
fig, axes = OutlierPlotFactory.create_boxplot_comparison(
    data=df_train,
    features=top_features[:6],
    target_col='outcome',
    n_cols=3,
    figsize=(15, 8)
)

plt.suptitle('Outlier Detection: Box Plots by Class', fontsize=16, weight='bold', y=1.00)
plt.show()

print("\n**Insight:** Outliers present but appear to be genuine sonar readings, not errors")
print("**Decision:** Keep outliers (real data), use robust scaling if needed")

## 9. Scaling Requirements

Assess need for feature scaling based on range differences.

In [ ]:
# Feature ranges
ranges = df_train[feature_cols].agg(['min', 'max', lambda x: x.max() - x.min()]).T
ranges.columns = ['Min', 'Max', 'Range']
ranges = ranges.sort_values('Range', ascending=False)

print("Feature Ranges (Top 10 widest):")
print(ranges.head(10))

print(f"\nOverall min: {ranges['Min'].min():.4f}")
print(f"Overall max: {ranges['Max'].max():.4f}")
print(f"Max range: {ranges['Range'].max():.4f}")
print(f"Range std: {ranges['Range'].std():.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(ranges)), ranges['Range'], color='#00847E', alpha=0.7)
ax.set_xlabel('Features (sorted by range)', fontsize=11)
ax.set_ylabel('Range', fontsize=11)
ax.set_title('Feature Value Ranges', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n**Insight:** Features have similar scales (0-1 range) but vary in distribution")
print("**Decision:** StandardScaler required for distance-based models (KNN, SVM)")
print("             Not required for tree-based models (Random Forest)")

## 10. Correlation Analysis

In [ ]:
# Feature-target correlations
target_corr = correlation_with_target(df_train, target='outcome')

print("Top 20 features by absolute correlation with target:")
print(target_corr.head(20))

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
target_corr.head(20).plot(kind='barh', ax=ax, color='#E3120B', alpha=0.7)
ax.set_xlabel('Absolute Correlation', fontsize=11)
ax.set_title('Top 20 Features by Target Correlation', fontsize=14, weight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Multicollinearity check
features_only = df_train.drop('outcome', axis=1)
corr_matrix = features_only.corr()

high_corr = count_high_correlations(corr_matrix, threshold=0.85)

print(f"Feature pairs with |correlation| > 0.85: {len(high_corr)}")
if len(high_corr) > 0:
    print("\nTop 10 highly correlated pairs:")
    print(high_corr.head(10))
    print("\n**Insight:** Some multicollinearity present")
    print("**Decision:** Ridge/Elastic Net regularization may help, or PCA for dimensionality reduction")
else:
    print("\n**Insight:** No severe multicollinearity detected")
    print("**Decision:** Standard models acceptable, regularization optional")

## 11. Class Signature Visualization (Radar Plot)

Visual representation of mean feature values by class.
Shows the unique "sonar fingerprint" of mines vs rocks.

In [ ]:
# Use all 60 features for comprehensive fingerprint
all_features = [c for c in df_train.columns if c != 'outcome']

fig, ax = RadarChartFactory.create_class_signature(
    data=df_train,
    features=all_features,
    target_col='outcome',
    class_0_label='Rock',
    class_1_label='Mine',
    title='Sonar Signature: Mean Feature Values by Class',
    figsize=(14, 14)
)

plt.show()

print("\n**Insight:** Distinct sonar signatures visible - mines and rocks have different frequency response patterns")
print("**Decision:** Classification is feasible based on distinct acoustic signatures")

## 12. Dimensionality Assessment (PCA)

In [ ]:
# Standardize features for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_only)

# Fit PCA
pca = PCA(n_components=min(30, len(feature_cols)))
pca_result = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scree plot
axes[0].bar(range(1, len(explained_var)+1), explained_var, 
           color='#00847E', alpha=0.7)
axes[0].set_xlabel('Principal Component', fontsize=11)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=11)
axes[0].set_title('PCA Scree Plot', fontsize=14, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative variance
axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var, 
            marker='o', color='#E3120B', linewidth=2)
axes[1].axhline(y=0.95, color='#363636', linestyle='--', 
               linewidth=2, label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=11)
axes[1].set_title('Cumulative Explained Variance', fontsize=14, weight='bold')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

n_components_95 = (cumulative_var >= 0.95).argmax() + 1
print(f"\nComponents needed for 95% variance: {n_components_95}")
print(f"Current dimensions: {len(feature_cols)}")
print(f"Reduction potential: {len(feature_cols) - n_components_95} features")

print("\n**Insight:** Moderate dimensionality - can be reduced but not excessive")
print("**Decision:** PCA optional - try both full features and PCA-reduced in modeling")

## 13. Class Separability (2D Projection)

In [ ]:
# 2D PCA projection
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#0F5499' if y == 0 else '#E3120B' for y in y_train]
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                    c=colors, alpha=0.6, edgecolors='k', s=100, linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.set_title('PCA 2D Projection: Mine vs Rock Separation', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#0F5499', label='Rock'),
    Patch(facecolor='#E3120B', label='Mine')
]
ax.legend(handles=legend_elements, frameon=False)

plt.tight_layout()
plt.show()

print(f"\nFirst 2 PCs explain {pca_2d.explained_variance_ratio_.sum():.1%} of variance")
print("\n**Insight:** Classes show partial separation with overlap")
print("**Decision:** Non-linear models (SVM RBF, Random Forest) likely needed")
print("             Linear models may struggle with overlapping regions")

## Key Findings Summary

### Data Quality
- ✓ No missing values
- ✓ Clean, continuous features
- ✓ Outliers present but genuine

### Dataset Characteristics
- **Size:** 166 training samples (small dataset)
- **Balance:** Relatively balanced classes
- **Features:** 60 sonar frequencies, similar scales
- **Signal:** Multiple strong predictors (correlations >0.4)

### Class Separability
- Distinct sonar signatures (radar plot)
- Partial separation in PCA space
- Some overlap - challenging but feasible

### Preprocessing Requirements
- StandardScaler: Required for KNN, SVM, LDA
- Imputation: Not needed (complete data)
- Resampling: Not needed (balanced classes)

---

## Modeling Strategy (Next Notebook)

### Constraints
1. **Small dataset (166 samples)** → Overfitting risk high
2. **Non-linear separability** → Need flexible models
3. **Recall priority** → Optimize for F2 score

### Approach
1. Use rigorous 5-fold stratified CV
2. Test diverse model families:
   - Linear: Logistic, Ridge, LDA
   - Non-parametric: KNN, Naive Bayes  
   - Non-linear: SVM (RBF), Random Forest
3. Hyperparameter tuning with GridSearchCV
4. Evaluate on F2 score (recall-weighted)
5. Select champion based on CV performance
6. Final test set evaluation
7. Threshold optimization for maximum recall

### Success Criteria
- Test F2 score >0.85
- Test recall >95%
- Stable CV performance (low std)

**Continue to Notebook 2: Model Training & Selection**